# 🌍 External Validation with Cross-Platform Harmonization
## Validating BCR Model on Independent Cohorts (e.g., GSE70769)

**Purpose:** Validate the trained model on external microarray data after proper harmonization.

### Key Steps:
1. Load external cohort (GSE70769 or similar)
2. Map gene symbols to TCGA format
3. Apply cross-platform normalization (quantile/z-score/ComBat)
4. Handle missing features (imputation or common-feature retraining)
5. Evaluate AUC, calibration, and clinical utility

---

In [ ]:
# Setup
import sys
sys.path.append('/workspace/core')

import pandas as pd
import numpy as np
import pickle
import logging
from src.io import setup_logging, logger
from src.data_loaders import load_external_gse70769
from src.validation import normalize_cross_platform, validate_features_available
from sklearn.metrics import roc_auc_score, average_precision_score, calibration_curve
import matplotlib.pyplot as plt

setup_logging(level=logging.INFO)
RANDOM_STATE = 42

## Step 1: Load Trained Model

In [ ]:
# Load the trained model from Official Pipeline
with open('/workspace/core/models/final_bcr_model.pkl', 'rb') as f:
    model_data = pickle.load(f)

final_model = model_data['model']
trained_features = model_data['features']
train_auc = model_data['genomic_auc']

logger.info(f"Model loaded with {len(trained_features)} features")
logger.info(f"Training AUC: {train_auc:.3f}")
logger.info(f"Features: {trained_features}")

## Step 2: Load External Cohort

In [ ]:
# Load GSE70769 (or other external cohort)
try:
    ext_clinical, ext_rna, ext_labels = load_external_gse70769()
    
    logger.info(f"External cohort: {len(ext_clinical)} patients, {ext_rna.shape[1]} genes")
    logger.info(f"BCR rate: {ext_labels.mean():.2%}")
    
    # Merge and align
    ext_merged = ext_clinical.merge(ext_rna, left_index=True, right_index=True, how='inner')
    X_ext = ext_merged.drop(columns=[c for c in ['BCR', 'days_to_bcr'] if c in ext_merged.columns], errors='ignore')
    y_ext = ext_labels.reindex(X_ext.index).dropna()
    X_ext = X_ext.loc[y_ext.index]
    
    logger.info(f"Final external dataset: {X_ext.shape[0]} samples")
    
except Exception as e:
    logger.error(f"Failed to load external data: {e}")
    logger.info("Creating synthetic external data for demonstration...")
    
    # Synthetic demo (replace with actual loading)
    from sklearn.datasets import make_classification
    X_syn, y_syn = make_classification(
        n_samples=100, n_features=len(trained_features),
        n_informative=5, random_state=RANDOM_STATE
    )
    X_ext = pd.DataFrame(X_syn, columns=trained_features)
    y_ext = pd.Series(y_syn)

## Step 3: Feature Mapping and Intersection

In [ ]:
# Check feature overlap
common_features = list(set(trained_features) & set(X_ext.columns))
missing_features = [f for f in trained_features if f not in X_ext.columns]

logger.info(f"Common features: {len(common_features)}/{len(trained_features)}")
if missing_features:
    logger.warning(f"Missing features ({len(missing_features)}): {missing_features}")

# Strategy: Use only common features
if len(common_features) < len(trained_features) * 0.8:
    logger.warning("Less than 80% feature overlap - consider retraining on common features only")
    
X_ext_common = X_ext[common_features]

## Step 4: Cross-Platform Normalization

In [ ]:
# Load training data for normalization reference
from src.data_loaders import load_tcga_prad_bcr
clinical_train, rna_seq_train, _ = load_tcga_prad_bcr()
merged_train = clinical_train.merge(rna_seq_train, left_index=True, right_index=True, how='inner')
X_train = merged_train[common_features].copy()

# Apply quantile normalization (recommended for cross-platform)
X_train_norm, X_ext_norm = normalize_cross_platform(
    df_train=X_train,
    df_test=X_ext_common,
    method='quantile',
    common_features=common_features
)

logger.info("Cross-platform normalization complete")
logger.info(f"Train mean: {X_train_norm[common_features].mean().mean():.4f}")
logger.info(f"External mean (after): {X_ext_norm[common_features].mean().mean():.4f}")

## Step 5: External Validation

In [ ]:
# Predict probabilities
# Note: If model was trained on all features, we need to retrain on common features
if len(common_features) == len(trained_features):
    prob_ext = final_model.predict_proba(X_ext_norm)[:, 1]
else:
    logger.warning("Retraining model on common features for external validation...")
    from src.improved_pipeline import build_elastic_net
    model_ext = build_elastic_net(random_state=RANDOM_STATE)
    model_ext.fit(X_train_norm, y.reindex(X_train.index))
    prob_ext = model_ext.predict_proba(X_ext_norm)[:, 1]

# Calculate metrics
auc_ext = roc_auc_score(y_ext, prob_ext)
ap_ext = average_precision_score(y_ext, prob_ext)

print("\n" + "="*60)
print("🌍 EXTERNAL VALIDATION RESULTS")
print("="*60)
print(f"Cohort: GSE70769 (example)")
print(f"Samples: {len(y_ext)}")
print(f"Features used: {len(common_features)}/{len(trained_features)}")
print(f"AUC: {auc_ext:.3f} (vs train {train_auc:.3f})")
print(f"AP:  {ap_ext:.3f}")
print(f"Drop: {train_auc - auc_ext:+.3f}")
print("="*60)

# Calibration plot
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
prob_true, prob_pred = calibration_curve(y_ext, prob_ext, n_bins=10)
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax.plot(prob_pred, prob_true, marker='o', label='Model')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title('Calibration Plot - External Validation')
ax.legend()
plt.tight_layout()
plt.savefig('/workspace/core/results/external_calibration.png', dpi=150)
plt.show()

logger.info("Calibration plot saved to /workspace/core/results/external_calibration.png")

## Step 6: Save External Validation Results

In [ ]:
import json

external_results = {
    'cohort': 'GSE70769',
    'n_samples': len(y_ext),
    'n_features_used': len(common_features),
    'n_features_total': len(trained_features),
    'missing_features': missing_features,
    'auc_external': auc_ext,
    'auc_training': train_auc,
    'auc_drop': train_auc - auc_ext,
    'average_precision': ap_ext
}

with open('/workspace/core/results/external_validation.json', 'w') as f:
    json.dump(external_results, f, indent=2)

logger.info("External validation results saved to /workspace/core/results/external_validation.json")

# Assessment
if auc_ext >= 0.60:
    logger.info("✅ SUCCESS: External AUC ≥ 0.60 - Model generalizes well!")
elif auc_ext >= 0.55:
    logger.warning("⚠️ MODERATE: External AUC 0.55-0.60 - Some generalization")
else:
    logger.error("❌ FAILURE: External AUC < 0.55 - Model does not generalize")
    logger.error("Recommendations:")
    logger.error("  1. Retrain on common features only")
    logger.error("  2. Try different normalization (ComBat instead of quantile)")
    logger.error("  3. Increase regularization")
    logger.error("  4. Consider platform-specific models")